In [1]:
import random
from pyspark import SparkConf
import json
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

import os
import sys
sys.path.append(os.path.abspath(".."))
from Models import Artist, Album, Track, DB_admin, Queries

In [2]:
#Configuramos la aplicacion de Spark
conf = SparkConf()


In [3]:
with open('/Users/axel/Documents/Portafolio/Spotify_api/PySpark/spark_conf.json') as spk:
    spark_config = json.load(spk)

conf.setAll(spark_config.items())
spark = (SparkSession.builder.config(conf=conf).getOrCreate())
print(spark.getActiveSession())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/02 16:20:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
#Creamos un df con spark de manea automatica 

eschema = StructType([
        StructField("id",IntegerType(),False),
        StructField("nombre",StringType(),True),
        StructField("edad",StringType(),True)
    ])

names = ["Sumi","Mozta","Minkie","Oliver","Werejero"]
data = [(id, random.choice(names), random.randint(5,30)) for id in range(1,11)]

df = spark.createDataFrame(data, schema=eschema)
df.show()

+---+--------+----+
| id|  nombre|edad|
+---+--------+----+
|  1|Werejero|  10|
|  2|   Mozta|  21|
|  3|    Sumi|  11|
|  4|  Minkie|  25|
|  5|Werejero|  10|
|  6|   Mozta|  19|
|  7|  Oliver|  15|
|  8|   Mozta|   5|
|  9|    Sumi|   8|
| 10|   Mozta|  18|
+---+--------+----+



In [5]:
#mandamos a traer una consulta leida de asqlalchemy
querry = await Queries.album_por_id_artista(2)

datos_limpios = [dict(fila) for fila in querry]

df_querry = spark.createDataFrame(datos_limpios)

df_querry.printSchema()

df_querry.show()

2026-08-28 11:14:27,223 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2026-08-28 11:14:27,223 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-08-28 11:14:27,254 INFO sqlalchemy.engine.Engine select current_schema()
2026-08-28 11:14:27,255 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-08-28 11:14:27,257 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2026-08-28 11:14:27,258 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-08-28 11:14:27,259 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-08-28 11:14:27,262 INFO sqlalchemy.engine.Engine SELECT artista.id_artista, artista.nombre_artista, album.id_album, album.nombre_album, album.id_spotify, album.fecha_lanzamiento, album.num_canciones 
FROM artista LEFT OUTER JOIN album ON artista.id_artista = album.id_artista 
WHERE artista.id_artista = $1::INTEGER ORDER BY album.id_album ASC
2026-08-28 11:14:27,263 INFO sqlalchemy.engine.Engine [generated in 0.00049s] (2,)
2026-08-28 11:14:27,273 INFO sqlalchemy.engine.En

In [5]:
path = '/Users/axel/Documents/Portafolio/Spotify_api/PySpark/macrodata.csv'

df = spark.read.csv(path, header=True)
df.printSchema()

df.show(random.randint(1,5))

df_dos = df.select("year","quarter","fecha")
df_dos.printSchema()

df_dos.show(random.randint(1,5))

root
 |-- year: string (nullable = true)
 |-- quarter: string (nullable = true)
 |-- realgdp: string (nullable = true)
 |-- realcons: string (nullable = true)
 |-- realinv: string (nullable = true)
 |-- realgovt: string (nullable = true)
 |-- realdpi: string (nullable = true)
 |-- cpi: string (nullable = true)
 |-- m1: string (nullable = true)
 |-- tbilrate: string (nullable = true)
 |-- unemp: string (nullable = true)
 |-- pop: string (nullable = true)
 |-- infl: string (nullable = true)
 |-- realint: string (nullable = true)
 |-- fecha: string (nullable = true)

+------+-------+--------+--------+-------+--------+-------+-----+-----+--------+-----+-------+----+-------+----------+
|  year|quarter| realgdp|realcons|realinv|realgovt|realdpi|  cpi|   m1|tbilrate|unemp|    pop|infl|realint|     fecha|
+------+-------+--------+--------+-------+--------+-------+-----+-----+--------+-----+-------+----+-------+----------+
|1959.0|    1.0|2710.349|  1707.4|286.898| 470.045| 1886.9|28.98|139.7| 

In [7]:
#Select funcionando como selector de columnas para modificacion y remapeo de estas
from pyspark.sql.functions import col
infor = (
    df.select(
        col("year").cast(DoubleType()),
        col("quarter").cast(DoubleType()),
        col("pop").alias("poblacion")
    )
)

print(infor)

infor.show()

DataFrame[year: double, quarter: double, poblacion: string]
+------+-------+---------+
|  year|quarter|poblacion|
+------+-------+---------+
|1959.0|    1.0|  177.146|
|1959.0|    2.0|   177.83|
|1959.0|    3.0|  178.657|
|1959.0|    4.0|  179.386|
|1960.0|    1.0|  180.007|
|1960.0|    2.0|  180.671|
|1960.0|    3.0|  181.528|
|1960.0|    4.0|  182.287|
|1961.0|    1.0|  182.992|
|1961.0|    2.0|  183.691|
|1961.0|    3.0|  184.524|
|1961.0|    4.0|  185.242|
|1962.0|    1.0|  185.874|
|1962.0|    2.0|  186.538|
|1962.0|    3.0|  187.323|
|1962.0|    4.0|  188.013|
|1963.0|    1.0|   188.58|
|1963.0|    2.0|  189.242|
|1963.0|    3.0|  190.028|
|1963.0|    4.0|  190.668|
+------+-------+---------+
only showing top 20 rows


In [8]:
infor_expre = df.selectExpr(
        "CAST(year as int) as year",
        "CAST(quarter as int) as quarter",
        "(pop) as poblacion"
    ).printSchema()


type(infor_expre)

root
 |-- year: integer (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- poblacion: string (nullable = true)



NoneType

In [9]:
#Metodo para withColum y withColumRenamed

#WithColum para agregar una nueva columna con un parametro por ejemplo dummy

df_dos.withColumn(
    "date_dummy_extract", 
    F.year(F.current_date()) - F.year('fecha').cast(IntegerType()) 
    ).show(5)

df_tres= df_dos.withColumn(
    "date_dummy_extract", 
    F.year(F.current_date()) - F.year('fecha').cast(IntegerType()) 
    )

#usamos el withColumnRenamed para unicamente renombrar la columna
df_tres.withColumnRenamed("date_dummy_extract","fecha-current_date").show(5)

+------+-------+----------+------------------+
|  year|quarter|     fecha|date_dummy_extract|
+------+-------+----------+------------------+
|1959.0|    1.0|1959-01-01|                67|
|1959.0|    2.0|1959-04-01|                67|
|1959.0|    3.0|1959-07-01|                67|
|1959.0|    4.0|1959-10-01|                67|
|1960.0|    1.0|1960-01-01|                66|
+------+-------+----------+------------------+
only showing top 5 rows
+------+-------+----------+------------------+
|  year|quarter|     fecha|fecha-current_date|
+------+-------+----------+------------------+
|1959.0|    1.0|1959-01-01|                67|
|1959.0|    2.0|1959-04-01|                67|
|1959.0|    3.0|1959-07-01|                67|
|1959.0|    4.0|1959-10-01|                67|
|1960.0|    1.0|1960-01-01|                66|
+------+-------+----------+------------------+
only showing top 5 rows


In [10]:
#Agrupacion y Agrupacion

df_querry.show()

+-----------------+--------+----------+--------------------+--------------------+--------------+-------------+
|fecha_lanzamiento|id_album|id_artista|          id_spotify|        nombre_album|nombre_artista|num_canciones|
+-----------------+--------+----------+--------------------+--------------------+--------------+-------------+
|       2024-11-22|      36|         2|0hvT3yIEysuuvkK73...|                 GNX|Kendrick Lamar|           12|
|       2022-05-13|      37|         2|79ONNoS4M9tfIA1mY...|Mr. Morale & The ...|Kendrick Lamar|           19|
|       2018-02-09|      38|         2|3pLdWdkj83EYfDN6H...|Black Panther The...|Kendrick Lamar|           14|
|       2017-12-08|      39|         2|4alcGHjstaALJHHil...|DAMN. COLLECTORS ...|Kendrick Lamar|           14|
|       2017-04-14|      40|         2|4eLPsYPBmXABThSJ8...|               DAMN.|Kendrick Lamar|           14|
|       2016-03-04|      41|         2|0kL3TYRsSXnu0iJvF...|untitled unmastered.|Kendrick Lamar|            8|
|

In [10]:
df_struct = StructType([
        StructField("duracion",StringType(),True),
        StructField("id_album",StringType(),True),
        StructField("id_artista",StringType(),True),
        StructField("id_cancion",StringType(),True),
        StructField("nombre_album",StringType(),True),
        StructField("nombre_artista",StringType(),True),
        StructField("nombre_cancion",StringType(),True),
        StructField("num_cancion",StringType(),True)
    ])


querry_2 = await Queries.querry_custom_spark()

datos_limpios_2 = [dict(fila) for fila in querry_2]

df_querry_2 = spark.createDataFrame(datos_limpios_2, schema=df_struct)

df_querry_2.show()


2026-09-02 16:41:50,133 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-02 16:41:50,136 INFO sqlalchemy.engine.Engine SELECT artista.id_artista, artista.nombre_artista, album.id_album, album.nombre_album, canciones.id_cancion, canciones.nombre_cancion, canciones.num_cancion, canciones.duracion 
FROM artista LEFT OUTER JOIN album ON artista.id_artista = album.id_artista LEFT OUTER JOIN canciones ON album.id_album = canciones.id_album ORDER BY canciones.id_cancion
2026-09-02 16:41:50,137 INFO sqlalchemy.engine.Engine [cached since 835.6s ago] ()
2026-09-02 16:41:50,153 INFO sqlalchemy.engine.Engine ROLLBACK
+--------+--------+----------+----------+--------------------+--------------+--------------------+-----------+
|duracion|id_album|id_artista|id_cancion|        nombre_album|nombre_artista|      nombre_cancion|num_cancion|
+--------+--------+----------+----------+--------------------+--------------+--------------------+-----------+
|  362068|       1|         1|         1|Unseen

In [20]:
df_filtrado = df_querry_2.filter(F.col("nombre_artista") == "Hocico") 

df_con_nucleo_nativo = df_filtrado.withColumn("nucleo_asignado", F.spark_partition_id())


print("==================================================")
print("   REPORTE: PARTICIONAMIENTO AUTOMÁTICO DE SPARK  ")
print("==================================================")

reporte_nativo = (
    df_con_nucleo_nativo.groupBy("nucleo_asignado")
    .agg(F.count("*").alias("Total_Registros"))
    .orderBy("nucleo_asignado")
)

reporte_nativo.show()

   REPORTE: PARTICIONAMIENTO AUTOMÁTICO DE SPARK  
+---------------+---------------+
|nucleo_asignado|Total_Registros|
+---------------+---------------+
|              0|            242|
|              1|            198|
+---------------+---------------+



In [ ]:
df_filtrado = df_querry_2.filter(F.col("nombre_artista") == "Hocico") 

df_particionado = df_filtrado.repartition(4, "id_artista", "id_album")


df_con_nucleo_nativo = df_particionado.withColumn("nucleo_asignado", F.spark_partition_id())


print("==================================================")
print("   REPORTE: PARTICIONAMIENTO AUTOMÁTICO DE SPARK  ")
print("==================================================")

reporte_nativo = (
    df_con_nucleo_nativo.groupBy("nucleo_asignado")
    .agg(F.count("*").alias("Total_Registros"))
    .orderBy("nucleo_asignado")
)

reporte_nativo.show()

   REPORTE: PARTICIONAMIENTO AUTOMÁTICO DE SPARK  
+---------------+---------------+
|nucleo_asignado|Total_Registros|
+---------------+---------------+
|              0|            176|
|              1|             85|
|              2|             45|
|              3|            134|
+---------------+---------------+



26/09/02 21:08:48 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 990540 ms exceeds timeout 120000 ms
26/09/02 21:08:48 WARN SparkContext: Killing executors is not supported by current scheduler.
26/09/02 21:08:50 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:34)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.stora

In [45]:
info = (
    df_querry_2.groupBy("id_artista").agg(
        F.count("duracion").alias("# canciones por artista"),
        F.sum("duracion").alias("Tiempo total de canciones por artista en milisegundos"),
        F.min("duracion").alias("cancion con duracion minima"),
        F.max("duracion").alias("cancion con duracion maxima")
    )
)

info.show()

+----------+-----------------------+-----------------------------------------------------+---------------------------+---------------------------+
|id_artista|# canciones por artista|Tiempo total de canciones por artista en milisegundos|cancion con duracion minima|cancion con duracion maxima|
+----------+-----------------------+-----------------------------------------------------+---------------------------+---------------------------+
|         1|                    440|                                            131267849|                      36333|                     582785|
|         2|                    155|                                             40485595|                      75535|                     727106|
|         3|                    411|                                            134303234|                      13752|                    1128800|
|         4|                    335|                                             70596364|                      37120|